# Logistic Regression

Let's now consider a second concrete model within the ERM framework. This time, the goal is to predict a **binary** outcome. For the sake of simplicity, we'll encode the two binary outcome values as
$$
y \in \{0,1\}
$$
However it should be noted that this encoding is pretty arbitrary (why not use any other two numbers, they're just as valid labels). As before our input is $x \in \mathbb{R}^D$.

We begin by choosing a class of score functions $\mathcal{S}$. Again, let's work with a model that is **linear in the parameters**:
$$
s_w(x) = w^\top x.
$$


Our interpretation of the score function is that higher values are associated with $y=1$ while smaller values are associated with $y=0$. 

The next step is to choose a **loss** function. In linear regression, we used squared error:
$$
\ell(y, s(x)) = (y-s(x))^2.
$$

While we *could* use this (its honestly not the stupidest thing to do) this is a bit weird because $y\in\{0,1\}$ is an arbitrary encoding of a binary label. Yet we're using it in algebraic ways (subtraction) as if that particular value/encoding of $y$ has a real *scale* meaning.

Here is a more commonly used loss in this binary case. Let $p_w(x) = \sigma(w^\top x) = \sigma(s(x))$ where $\sigma$ is the logistic **link** or **activation** function:

$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$

Note that $p_w(x)$ gives outputs in $(0,1)$, which seems to make more sense for binary prediction. Furthermore its an increasing function so:

- if $w^\top x \gg 0$, then $\sigma(w^\top x) \approx 1$
- if $w^\top x \ll 0$, then $\sigma(w^\top x) \approx 0$
- if $w^\top x = 0$, then $\sigma(w^\top x) = 1/2$

So the quantity $w^\top x$ is still doing the linear modeling work, but we are now mapping that score onto a bounded scale. One can think of $p_w(x)$ as giving a *fuzzy* (probabilistic?) estimate of $y$. 

Now that we have a way to map our linear score into $(0,1)$, we need to decide how to measure error.

A natural idea is:

- if $y = 1$, we want $p_w(x)$ to be close to 1  
- if $y = 0$, we want $p_w(x)$ to be close to 0  

So we want a loss that:

- is small when $p_w(x)$ matches $y$  
- is large when it does not  

A commonly used choice that satisfies these properties is the **cross-entropy loss**:
$$
\ell(y, s_w(x)) = -\Big(y \log \sigma(s_w(x)) + (1-y)\log(1 - \sigma(s_w(x)))\Big).
$$
With a bit of algebra, this simplifies to a very clean expression:
$$
\ell(y, s_w(x)) = \log(1 + e^{s_w(x)}) - y\, s_w(x).
$$
All of this is essentially equivalent to:
$$
\ell(y, s(x)) = -\Big(y \log p_w(x) + (1-y)\log(1 - p_w(x))\Big).
$$
since $p_w$ is a function of $s(x)$.


### Understanding the loss

It is helpful to look at the two cases separately.

- If $y = 1$:
$$
\ell(1, p) = -\log p
$$

  - Small when $p \approx 1$  
  - Very large when $p \approx 0$

- If $y = 0$:
$$
\ell(0, p) = -\log(1 - p)
$$

  - Small when $p \approx 0$  
  - Very large when $p \approx 1$

So:

- correct predictions → small loss  
- incorrect but middling predictions $p\approx .5$ → moderate loss  
- incorrect and "confident" predictions → **very large loss**

This last point is important. The loss strongly discourages being confidently wrong.

### Why not squared error?

We could use squared error here:
$$
\ell(y, s(x)) = (y - \sigma(s(x)))^2,
$$
but it is not a great fit for this setting.

- Squared error treats the problem as if the labels 0 and 1 have a meaningful numerical scale  
- It does not as strongly penalize confident mistakes  
- It leads to a non-convex optimization problem when combined with the sigmoid when optimizing for $w$

Lets plot these losses:

In [ ]:
#| code-fold: true

import numpy as np
import matplotlib.pyplot as plt

def cross_entropy(y, p):
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def squared_error(y, p):
    return (y - p) ** 2

# values of p = sigma(s)
p_vals = np.linspace(0.001, 0.999, 200)

# compute losses
ce_y1 = cross_entropy(1, p_vals)
se_y1 = squared_error(1, p_vals)

ce_y0 = cross_entropy(0, p_vals)
se_y0 = squared_error(0, p_vals)

# plot
plt.figure()

# y = 1
plt.plot(p_vals, ce_y1, label="CE (y=1)")
plt.plot(p_vals, se_y1, linestyle='--', label="SE (y=1)")

# y = 0
plt.plot(p_vals, ce_y0, label="CE (y=0)")
plt.plot(p_vals, se_y0, linestyle='--', label="SE (y=0)")

plt.xlabel("p = σ(s)")
plt.ylabel("Loss")
plt.title("Cross-Entropy vs Squared Error")
plt.legend()
plt.grid(True)

plt.show()

### ERM

Now we have everything we need for our ERM formulation. Putting everything together, we choose $w$ to minimize the empirical loss:
$$
\hat{R}(w) = \frac{1}{N} \sum_{n=1}^N
-\Big(y \log \sigma(w^\top x) + (1-y)\log(1 - \sigma(w^\top x))\Big).
$$
our goal is to find

$$
\hat{w} = \arg\min_{w} \hat{R}(w) = \arg\min_w \frac{1}{N} \sum_{n=1}^N
-\Big(y \log \sigma(w^\top x) + (1-y)\log(1 - \sigma(w^\top x))\Big).
$$

As with least squares, the factor $1/N$ does not affect the minimizer, so we often drop it when writing the optimization problem.

At this point, the setup should feel very similar to linear regression: we choose a model class, we choose a loss, and then we minimize empirical risk over $w$. The main difference is that the loss is now not so simple in $w$, so the optimization problem is no longer something we can solve in one line by setting a gradient equal to zero and rearranging.



## Vector Form

Let $X \in \mathbb{R}^{N \times D}$ be the design matrix, with row $n$ equal to $x_n^\top$, and let
$$
y = (y_1,\dots,y_N)^\top.
$$

Then the vector of linear scores is
$$
Xw \in \mathbb{R}^N,
$$
and let the vector $p$ be 
$$
p = \sigma(Xw),
$$
where the sigmoid is applied elementwise.

So
$$
p_n = \sigma(x_n^\top w).
$$

In this notation, the empirical risk is

$$
\hat{R}(w) =
-\frac{1}{N}
\sum_{n=1}^N
\Big(
y_n \log p_n + (1-y_n)\log(1-p_n)
\Big).
$$



## No Closed-Form Solution

For least squares, the objective was quadratic in $w$, which led to the normal equations:
$$
\hat{w} = (X^\top X)^{-1} X^\top y
$$
whenever $X^\top X$ was invertible.

Here, that does not happen. Because of the sigmoid and the logarithms, the objective is no longer quadratic in $w$. There is no analogous closed-form expression for $\hat{w}$.

That means we need to solve the problem using an **iterative optimization method**.

The two most important ones to discuss at this stage are:

- **gradient descent**, which uses first-order information
- **Newton's method**, which uses second-order information



## Gradient Descent

The idea of gradient descent is exactly the same as in ordinary calculus. If we want to minimize a smooth function, we compute its gradient and then move a small amount in the opposite direction. After all, the gradient (locally) points in the direction of steepest ascent in the function, so moving opposite of it points in the direction of quickest descent. 

To make this concrete, suppose we want to minimize some function:
$$
\hat{R}(w)
$$
where $w \in \mathbb{R}^D$.

Gradient descent proceeds iteratively. Starting from some initial guess $w^{(0)}$, we repeat the following steps:

1. **Compute the gradient at the current point**
$$
\nabla \hat{R}(w^{(t)})
$$

2. **Take a step in the opposite direction**
$$
w^{(t+1)} = w^{(t)} - \eta \, \nabla \hat{R}(w^{(t)})
$$

We repeat this until "convergence" e.g. when our value of $w^{(t)}$ stops changing much.

Here $\eta > 0$ is called the **step size** or **learning rate** (It controls how far we move in the descent direction). The choice of $\eta$ is important:

- If $\eta$ is too small:
  - progress is very slow  
- If $\eta$ is too large:
  - we may overshoot and fail to converge  

In practice, $\eta$ is often chosen by experimentation or adjusted over time.

So the key step is to compute the gradient $\nabla \hat{R}(w)$.


To do this, first, we can show that 
$$
\sigma'(t) = \sigma(t)(1-\sigma(t)).
$$

Now let
$$
p_n = \sigma(x_n^\top w).
$$

If we differentiate the $n$-th summand:
$$
\ell_n = -\Big(y_n \log p_n + (1-y_n)\log(1-p_n)\Big)
$$

and apply the **chain rule**, the algebra simplifies nicely and gives

$$
\nabla_w \ell_n =(p_n - y_n)x_n.
$$

Summing over all $n$, we obtain

$$
\nabla \hat{R}(w) = \frac{1}{N}\sum_{n=1}^N (p_n-y_n)x_n.
$$

In matrix form, this is

$$
\nabla \hat{R}(w)=\frac{1}{N}X^\top (p-y).
$$

This formula is worth comparing directly with linear regression. In least squares, the gradient involved the residual $Xw-y$. Here, it involves something like a "residual" $p-y$.



Consequently, gradient descent takes the form

$$
w^{(t+1)}=w^{(t)} - \eta \nabla \hat{R}(w^{(t)})=w^{(t)} - \eta \frac{1}{N}X^\top(p^{(t)}-y),
$$
where $\eta > 0$ is the learning rate.

So each iteration is:

1. compute the current "logits" $Xw^{(t)}$
2. apply the sigmoid to get $p^{(t)}$
3. compute the gradient $\frac{1}{N}X^\top(p^{(t)}-y)$
4. update $w$


Let's try implementing this in code:

In [ ]:
def sigmoid(z):
    z = np.clip(z, -50, 50)  # hacky numerical guard
    return 1.0 / (1.0 + np.exp(-z))

def cross_entropy_loss(X, y, w):
    p = sigmoid(X @ w)

    # hacky clipping for numerical stability
    eps = 1e-12
    p = np.clip(p, eps, 1 - eps)

    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def logistic_gradient(X, y, w):
    p = sigmoid(X @ w)
    return (X.T @ (p - y)) / len(y)

def gradient_descent_with_path(X, y, w0, lr=0.1, n_steps=1000, tol=1e-6):
    w = w0.copy()
    path = [w.copy()]
    losses = [cross_entropy_loss(X, y, w)]

    for _ in range(n_steps):
        grad = logistic_gradient(X, y, w)
        w_new = w - lr * grad

        # stopping condition
        if np.linalg.norm(w_new - w) < tol:
            w = w_new
            path.append(w.copy())
            losses.append(cross_entropy_loss(X, y, w))
            break

        w = w_new
        path.append(w.copy())
        losses.append(cross_entropy_loss(X, y, w))

    return w, np.array(losses), np.array(path)

Let's generate some really simple data:

In [ ]:
np.random.seed(12390)

n = 100

# Class 1
X_pos = np.random.randn(n//2, 2) + np.array([1.0, 1.0])

# Class 0
X_neg = np.random.randn(n//2, 2) + np.array([-1.0, -1.0])

# Stack together
X = np.vstack([X_pos, X_neg])
y = np.hstack([np.ones(n//2), np.zeros(n//2)])

print(X[:10])
print(y[45:55])

In [ ]:
plt.figure(figsize=(5,5))

plt.scatter(X_pos[:,0], X_pos[:,1], label="y = 1")
plt.scatter(X_neg[:,0], X_neg[:,1], label="y = 0")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("2D dataset")
plt.legend()
plt.grid(True)
plt.axis("equal")

plt.show()

Now we can run the gradient descent:

In [ ]:
w0 = np.array([0.0, 0.0])
w_hat, losses, path = gradient_descent_with_path(X, y, w0, lr=0.1, n_steps=10000, tol=1e-4)

In [ ]:
print("Final weights:", w_hat)
print("Final loss:", losses[-1])
print("Actual num steps:",len(path))

We can plot the loss v. iteration:

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses, marker="o", markersize=3)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Loss during gradient descent")
plt.grid(True)
plt.show()

We can also plot the path on the surface:

In [ ]:
w1_vals = np.linspace(-3, 3, 200)
w2_vals = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1_vals, w2_vals)

Z = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        w = np.array([W1[i, j], W2[i, j]])
        Z[i, j] = cross_entropy_loss(X, y, w)

plt.figure(figsize=(7, 6))
contours = plt.contour(W1, W2, np.log(Z), levels=25)
plt.clabel(contours, inline=True, fontsize=8)

plt.plot(path[:, 0], path[:, 1], marker="o", markersize=3)
plt.scatter(path[0, 0], path[0, 1], s=80, marker="x", label="start")
plt.scatter(path[-1, 0], path[-1, 1], s=80, marker="*", label="end")

plt.xlabel(r"$w_1$")
plt.ylabel(r"$w_2$")
plt.title("Loss surface and gradient descent path")
plt.legend()
plt.grid(True)
plt.show()

## Newton's Method

Gradient descent uses only slope information. Newton's method goes one step further: it also uses **curvature**.

The basic idea behind Newton’s method is to use a **local quadratic approximation** of the objective to choose a better update direction.

To see where this comes from, recall the second-order Taylor approximation in one dimension. If $f:\mathbb{R}\to\mathbb{R}$ is smooth, then near a point $w$,
$$
f(w + \Delta)
\approx
f(w)
+
f'(w)\,\Delta
+
\frac{1}{2} f''(w)\,\Delta^2.
$$

This is just saying: locally, a smooth function looks like a **quadratic curve**.

In higher dimensions, the same idea holds. For our objective $\hat{R}(w)$, we have the approximation:

$$
\hat{R}(w+\Delta)
\approx
\hat{R}(w)
+
\nabla \hat{R}(w)^\top \Delta
+
\frac{1}{2}\Delta^\top H(w)\Delta,
$$

where:

- $\nabla \hat{R}(w)$ is the gradient  
- $H(w)$ is the Hessian (matrix of all mixed partial second derivatives):

$$
H(w)_{ij} = \frac{\partial^2 \hat{R}(w)}{\partial w_i \partial w_j}.
$$

So locally, we are replacing our complicated objective with a **quadratic function in $\Delta$**.

Now, instead of minimizing $\hat{R}$ directly, we minimize this quadratic approximation. This is much easier, since it has a closed-form solution.

Taking the gradient with respect to $\Delta$ and setting it equal to zero:
$$
\nabla \hat{R}(w) + H(w)\Delta = 0,
$$
which gives:
$$
\Delta^\star = -H(w)^{-1}\nabla \hat{R}(w).
$$

We use a *Newton* update in place of the standard gradient descent with $\eta$:

$$
w \leftarrow w + \Delta^\star = w - H(w)^{-1}\nabla \hat{R}(w).
$$

with $H(w)^{-1}$ playing the place of $\eta$.

Intuition:

- Gradient descent uses only **first-order information** (slope)  
- Newton's method also uses **second-order information** (curvature)  

The Hessian tells us how the function is curved, so:

- we take **smaller steps** in steep directions  
- and **larger steps** in flatter directions  

As a result, Newton’s method often converges in far fewer iterations than gradient descent (though each step is more expensive).


For logistic regression, the Hessian also has a clean form. Let
$$
p = \sigma(Xw),
$$
and define the diagonal matrix
$$
W = \mathrm{diag}(p_1(1-p_1), \dots, p_N(1-p_N)).
$$

Then
$$
\nabla^2 \hat{R}(w) = H(w) = 
\frac{1}{N}X^\top W X.
$$

So the Newton step becomes
$$
w^{(t+1)}=
w^{(t)} - (X^\top W^{(t)} X)^{-1}X^\top(p^{(t)}-y).
$$

A few remarks:

- each step is typically much more expensive than a gradient descent step
- but Newton's method often needs far fewer iterations
- the matrix $W$ changes from one iteration to the next

**Fun-fact**: This is sometimes rewritten as (what is called) a **weighted** least-squares problem, leading to what people call the Iteratively Re-weighted Least-Squares (IRLS) algorithm for fitting logistic regression. This is effectively Newton's method that we present here.


Let's look at it in code:

In [ ]:
def logistic_hessian(X, y, w):
    p = sigmoid(X @ w)
    W = p * (1 - p)   # diagonal entries only
    return (X.T @ (W[:, None] * X)) / len(y) # some hax to avoid large mtx multiplications

def newton_method_with_path(X, y, w0, n_steps=50, tol=1e-6):
    w = w0.copy()
    path = [w.copy()]
    losses = [cross_entropy_loss(X, y, w)]

    for _ in range(n_steps):
        grad = logistic_gradient(X, y, w)
        H = logistic_hessian(X, y, w)

        # Solve H delta = grad, then update w <- w - delta
        delta = np.linalg.solve(H, grad)
        w_new = w - delta

        path.append(w_new.copy())
        losses.append(cross_entropy_loss(X, y, w_new))

        if np.linalg.norm(w_new - w) < tol:
            w = w_new
            break

        w = w_new

    return w, np.array(losses), np.array(path)

We can run the Newton's method:

In [ ]:
w0 = np.array([0.0, 0.0])
w_hat, losses, path = newton_method_with_path(X, y, w0, n_steps=10000, tol=1e-4)

print("Final weights:", w_hat)
print("Final loss:", losses[-1])
print("Number of steps:", len(losses) - 1)

And then plot loss v. iteration:

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses, marker="o", markersize=4)
plt.xlabel("Iteration")
plt.ylabel("Cross-entropy loss")
plt.title("Loss during Newton's method")
plt.grid(True)
plt.show()

In [ ]:
w1_vals = np.linspace(-3, 3, 200)
w2_vals = np.linspace(-3, 3, 200)
W1, W2 = np.meshgrid(w1_vals, w2_vals)

Z = np.zeros_like(W1)
for i in range(W1.shape[0]):
    for j in range(W1.shape[1]):
        w = np.array([W1[i, j], W2[i, j]])
        Z[i, j] = cross_entropy_loss(X, y, w)

plt.figure(figsize=(7, 6))
contours = plt.contour(W1, W2, np.log(Z), levels=25)
plt.clabel(contours, inline=True, fontsize=8)

plt.plot(path[:, 0], path[:, 1], marker="o", markersize=3)
plt.scatter(path[0, 0], path[0, 1], s=80, marker="x", label="start")
plt.scatter(path[-1, 0], path[-1, 1], s=80, marker="*", label="end")

plt.xlabel(r"$w_1$")
plt.ylabel(r"$w_2$")
plt.title("Loss surface and gradient descent path")
plt.legend()
plt.grid(True)
plt.show()

After fitting $w$, we still need to convert the score into a binary decision. Recall that before, our action for a classifier was
$$
a(s) = \arg\max_k s_k(x),
$$
where $s(x)$ was a $K$-component vector giving scores for each class. What happened to that!? We've only been working with one score function $s(x) = w^\top x$. 

We could have framed things like this: 
$$
s(x) = (s_1(x), s_2(x)), \quad s_k(x) = w_k^\top x,
$$
and predict:
$$
\hat{y} = \arg\max\{s_1(x), s_2(x)\}.
$$

We would then need to learn both $w_1$ and $w_2$ under some similar model. However, this parameterization is **over-complete**: only the difference between the scores matters. In particular, our argmax rule really boils down to:
$$
\hat{y} = 1 \quad \text{if } s_1(x) \ge s_2(x)
\quad \Longleftrightarrow \quad
s_1(x) - s_2(x) \ge 0.
$$

So the model depends only on the quantity $s_1(x) - s_2(x) = w_1^\top x - w_2^\top x = (w_1 - w_2)^\top x = w^\top x$ where $w = w_1-w_2$. We can therefore reparameterize the model by defining a single score:
$$
s_w(x) = w^\top x,
$$
which you can think of as representing the difference $s_1(x) - s_2(x)$. Equivalently, we can fix one class as a **reference classs** (here we'll use class 0) by setting:
$$
s_2(x) = 0, \quad s_1(x) = w^\top x.
$$

This removes the redundancy and leaves us with a single parameter vector $w$.

In any case, the prediction rule becomes:
$$
\hat{y} =
\begin{cases}
1 & \text{if } w^\top x \ge 0 \\
0 & \text{otherwise}.
\end{cases}
$$

So:

- we compute the linear score $s_w(x) = w^\top x$  
- we threshold at 0 to make a binary decision  

An equivalent way of thinking about this is considering $p_w$. During training, we introduced:
$$
p_w(x) = \sigma(w^\top x),
$$
which maps the score into $(0,1)$ (a probability!?). After this transformation, at prediction time, this corresponds to:
$$
\hat{y} = 1 \quad \text{if } p_w(x) \ge \tfrac{1}{2}.
$$

If we were so bold as to interpret this as a probability, this would seem reasonable.

We say that the **decision** boundary is where we go from predicting one class to another. For logistic regression, this decision boundary is defined by all $x$ such that 
$$
\hat{w}^\top x = 0.
$$
This is **linear**. 


We can plot the decision boundary learned from the previous problem:

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(X_pos[:, 0], X_pos[:, 1], label="y = 1")
plt.scatter(X_neg[:, 0], X_neg[:, 1], label="y = 0")

# Decision boundary: w1 x1 + w2 x2 = 0
if abs(w_hat[1]) > 1e-12:
    x1_vals = np.array([X[:, 0].min() - 1, X[:, 0].max() + 1])
    x2_vals = -(w_hat[0] / w_hat[1]) * x1_vals
    plt.plot(x1_vals, x2_vals, label=r"$w^\top x = 0$")
else:
    x_const = 0.0
    if abs(w_hat[0]) > 1e-12:
        x_const = 0.0
    plt.axvline(x=x_const, label=r"$w^\top x = 0$")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Data and learned decision boundary")
plt.legend()
plt.grid(True)
plt.axis("equal")
plt.show()


## What is Allowed in the Design Matrix?

The answer is essentially the same as in linear regression.

The design matrix $X$ determines how inputs are represented. Typically, its columns are measured covariates, but they can also be engineered features derived from those covariates.

The key point is the same as before:

> Logistic regression decision boundary is linear in the parameters $w$, not necessarily linear in the original raw inputs.

So we are free to include many kinds of columns in $X$, provided the representation makes sense for the problem.


## Limitations

Logistic regression is simple, useful, and often surprisingly effective. Still, it has several important limitations.

### Linear Decision Boundary

The model produces a linear decision boundary in feature space:
$$
w^\top x = 0.
$$

So if the classes are separated by a complicated nonlinear boundary, basic logistic regression may not perform well unless we introduce additional features. Because the decision boundary is linear in the chosen features, the performance of the model can depend heavily on how we construct the design matrix.

### Optimization Issues

Unlike least squares, there is no closed-form solution. We need an iterative optimization method such as gradient descent or Newton's method. One issue that can arise with logistic regression is when the data is **perfectly linearly separable**.

This means there exists some vector $w$ such that:
$$
y_i = 1 \implies w^\top x_i > 0, \quad
y_i = 0 \implies w^\top x_i < 0.
$$

In this case, the model can classify all points correctly with a linear decision boundary. While this seems good (maybe too good to be true), it can cause some numeric weirdness, which may get reported to the user. 

What weirdness can happen?

> If the above works for a vector $w$ then it also works for a vector $cw$ for some constant $c$. 


**No minimum achieved.** Recall the loss is representable as:
$$
\ell(y, s) = \log(1 + e^{s}) - y\,s.
$$

If the data is separable, we can scale $w$ by a large constant $c$:
$$
w \mapsto c\,w.
$$

Then:

- for $y=1$, $w^\top x \to +\infty$  
- for $y=0$, $w^\top x \to -\infty$  

This drives the loss toward zero:
$$
\ell(y_i, w^\top x_i) \to 0 \quad \text{for all } i.
$$

### Case 1: $y = 1$

Then the loss is:
$$
\ell(1, s) = \log(1 + e^{s}) - s.
$$

Plugging in $s = c\,w^\top x$:
$$
\ell(1, c\,w^\top x)
= \log(1 + e^{c\,w^\top x}) - c\,w^\top x.
$$

If the point is correctly classified, then $w^\top x > 0$, so as $c \to \infty$:

- $e^{c\,w^\top x}$ becomes very large  
- $\log(1 + e^{c\,w^\top x}) \approx c\,w^\top x$ 

So:
$$
\ell(1, c\,w^\top x)
\approx c\,w^\top x - c\,w^\top x = 0.
$$

### Case 2: $y = 0$

Then the loss is:
$$
\ell(0, s) = \log(1 + e^{s}).
$$

Plugging in $s = c\,w^\top x$:
$$
\ell(0, c\,w^\top x)
= \log(1 + e^{c\,w^\top x}).
$$

If the point is correctly classified, then $w^\top x < 0$, so as $c \to \infty$:

- $c\,w^\top x \to -\infty$  
- $e^{c\,w^\top x} \to 0$  

So:
$$
\ell(0, c\,w^\top x)
\approx \log(1 + 0) = 0.
$$

The key issue is:

> The loss can always be made smaller by increasing the magnitude of $w$.

So there is **no finite minimizer** of the objective and an infinite number of solutions that give a perfect classification. Since our loss function is on $s$ not $f$, even though many different vectors $w$ correctly separate the data, logistic regression prefers larger and larger margins. Sometimes, methods will throw an error or warning if they sense that this is happening. 

# Code Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.inspection import DecisionBoundaryDisplay

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (6, 4)

## Penguins

Let's work with the [penguins](https://allisonhorst.github.io/palmerpenguins/) dataset. The dataset is loaded directly from the CSV gist in the below code.

In [ ]:
url = "https://gist.githubusercontent.com/slopp/ce3b90b9168f2f921784de84fa445651/raw/penguins.csv"
penguins = pd.read_csv(url)
# clean up and shuffle
penguins = penguins.dropna().copy().sample(frac=1)

# for ease of visualization, let's subset down to two vars + species
cols = ["flipper_length_mm", "bill_depth_mm", "species"]
penguins = penguins[cols]
penguins.head()

**Binary logistic regression: Adelie vs Chinstrap**

First, we subset down to two classes, for binary logistic regression:

In [ ]:
# subset down to only Adelie and Chinstrap
d = penguins.loc[penguins["species"].isin(["Adelie", "Chinstrap"]), cols].copy()

d["species"].value_counts()

In [ ]:
d.sample(n=5)

In [ ]:
fig, ax = plt.subplots()
sns.scatterplot(
    data=d,
    x="flipper_length_mm",
    y="bill_depth_mm",
    hue="species",
    ax=ax,
)
ax.set_title("Adelie vs Chinstrap")
plt.show()

**Fit a logistic regression model**

We use the two numeric features:

- `flipper_length_mm`
- `bill_depth_mm`

and predict whether a penguin is Chinstrap (`y=1`) or Adelie (`y=0`). By default, sklearn orders the factors in alphabetical order.

In [ ]:
X_train = d[["flipper_length_mm","bill_depth_mm"]]
y_train = d["species"]

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
log_mod = LogisticRegression(C=np.inf)
log_mod.fit(X_train, y_train)

In [ ]:
log_mod.get_params()

For now we use this hack `C=np.inf`, which we'll discuss later. We can ignore the wanring for now.

**Scores, probabilities, and predictions**

- `decision_function` gives the linear score
- `predict_proba` gives class probabilities
- `predict` thresholds those probabilities at 0.5 in the binary case

In [ ]:
scores = log_mod.decision_function(X_train)
scores[:10]

In [ ]:
probs = log_mod.predict_proba(X_train)
probs[:10]

In [ ]:
preds = log_mod.predict(X_train)
preds[:10]

**Confusion matrix**

A **confusion matrix** is a simple way to summarize how well a classifier is performing.

For a binary classification problem, it is a $2 \times 2$ table that compares:
- the **true labels** (rows)
- the **predicted labels** (columns)

It has four entries:

- **True positives (TP):** predicted 1, actually 1  
- **False positives (FP):** predicted 1, actually 0  
- **True negatives (TN):** predicted 0, actually 0  
- **False negatives (FN):** predicted 0, actually 1  

So the matrix looks like:
$$
\begin{array}{c|cc}
 & \text{Pred 0} & \text{Pred 1} \\
\hline
\text{True 0} & \text{TN} & \text{FP} \\
\text{True 1} & \text{FN} & \text{TP}
\end{array}
$$

The confusion matrix gives a more detailed picture than just overall accuracy. It lets you see:

- how many mistakes the model makes  
- what *kind* of mistakes (false positives vs false negatives)  
- whether the model is biased toward one class  

This is especially important when the classes are imbalanced or when different types of errors have different costs.

In [ ]:
confusion_matrix(y_train, preds)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_train, preds),
    display_labels=["Adelie", "Chinstrap"],
).plot(ax=ax, colorbar=False)
ax.grid(False)
plt.show()

**Precision, recall, and F1 score**

Using the confusion matrix, we can define several useful metrics that summarize different aspects of performance.

**Precision**

**Precision** measures how accurate the positive predictions are:
$$
\text{Precision} = \frac{\text{TP}}{\text{TP} + \text{FP}}.
$$

- Of all points predicted as class 1, how many are actually class 1?  
- High precision means **few false positives**

**Recall**

**Recall** (also called sensitivity) measures how well we capture the positive class:
$$
\text{Recall} = \frac{\text{TP}}{\text{TP} + \text{FN}}.
$$

- Of all true class 1 points, how many did we correctly identify?  
- High recall means **few false negatives**

**F1 score**

The **F1 score** combines precision and recall into a single number:
$$
\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}.
$$

- It is the **harmonic mean** of precision and recall  
- It is high only when **both precision and recall are high**


We can get these in python with the `classification_report` function:

In [ ]:
print(classification_report(y_train, preds, target_names=["Adelie", "Chinstrap"]))

We should be able to get our same results as earlier, given our `newton_method_with_path` function:

In [ ]:
X_design = np.column_stack([np.ones(len(X_train)), X_train])
y01 = (y_train=="Chinstrap")*1 #Adelie = 0 (base), Chinstrap=1

In [ ]:
out = newton_method_with_path(X_design,y01,w0=np.zeros(3))

In [ ]:
out[0]

Recall that our previous results were:

In [ ]:
print(log_mod.intercept_)
print(log_mod.coef_)

Let's plot the decision regions

In [ ]:
def plot_binary_boundary(model, df, title, ax=None, palette = {
        "Adelie": "#1f77b4",
        "Chinstrap": "#ff7f0e",
    }):
    if ax is None:
        fig, ax = plt.subplots()

    X_plot = df[["flipper_length_mm", "bill_depth_mm"]]

    from matplotlib.colors import ListedColormap
    cmap = ListedColormap([palette[c] for c in model.classes_])

    DecisionBoundaryDisplay.from_estimator(
        model,
        X_plot,
        response_method="predict",
        cmap=cmap,
        alpha=0.25,
        ax=ax,
        eps=0.1,
        grid_resolution=300
    )

    sns.scatterplot(
        x=df["flipper_length_mm"],
        y=df["bill_depth_mm"],
        hue=df["species"],
        palette=palette,  
        ax=ax,
        edgecolor="black",
        s=60,
    )

    ax.set_title(title)
    return ax

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(log_mod, d, "Linear logistic regression", ax=ax)
plt.show()

Like linear regression, we can do all sorts of feature engineering if we want. For example, we can include second-degree polynomial terms if we want:

In [ ]:
quad_mod = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=2, include_bias=False),
    LogisticRegression(C=np.inf)
)
quad_mod.fit(X_train, y_train)

Under the hood a bit:

In [ ]:
logreg_mod = quad_mod.named_steps["logisticregression"]

In [ ]:
print(logreg_mod.intercept_)
print(logreg_mod.coef_)

In [ ]:
scaler = quad_mod.named_steps["standardscaler"]
poly = quad_mod.named_steps["polynomialfeatures"]

In [ ]:
print(scaler.mean_)
print(scaler.var_)

In [ ]:
X_train[:5]

In [ ]:
scaler.transform(X_train)[:5]

In [ ]:
poly.transform(scaler.transform(X_train)[:5])

In any case, we can plot the boundary:

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(quad_mod, d, "Quadratic logistic regression", ax=ax)
plt.show()

## A separable Case

In [ ]:
d_sep = penguins.loc[penguins["species"].isin(["Adelie", "Gentoo"]), cols].copy()

In [ ]:
X_sep = d_sep[["flipper_length_mm", "bill_depth_mm"]]
y_sep = d_sep["species"]

In [ ]:
sep_mod = LogisticRegression(C=np.inf)
sep_mod.fit(X_sep, y_sep)

In [ ]:
pal = {
        "Adelie": "#1f77b4",
        "Gentoo": "#ff7f0e",
    }
fig, ax = plt.subplots(figsize=(6, 5))
plot_binary_boundary(sep_mod, d_sep, "Linear logistic regression", ax=ax, palette=pal)
plt.show()

We don't get an error. What gives?

Warnings signs: predicted probability are very near 0/1 for everything

In [ ]:
sep_mod.predict_proba(X_sep)[:10]

accuracy is numerically `1`

In [ ]:
sep_mod.score(X_sep, y_sep)

If we reduce the tolerance the coefficients tend to blow up as we try harder to reduce the loss:

In [ ]:
# default tolerance
LogisticRegression(C=np.inf, tol=1E-4, max_iter=100).fit(X_sep, y_sep).coef_

In [ ]:
import warnings

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    cfs = []
    tols = 10**np.linspace(-4, -15, 100)
    
    for tl in tols:
        cf = LogisticRegression(C=np.inf, tol=tl).fit(X_sep, y_sep).coef_.ravel()
        cfs.append(np.append(cf, tl))  # append tol to coefficients
    
    cfs = pd.DataFrame(np.array(cfs),columns=["w1","w2","tol"])

Norm $||w||$ gets larger as tolerance decreases:

In [ ]:
plt.plot(np.log10(cfs['tol']),np.sqrt(cfs['w1']**2+cfs['w2']**2))
plt.scatter(np.log10(cfs['tol']),np.sqrt(cfs['w1']**2+cfs['w2']**2))
plt.xlabel("Tolerance")
plt.ylabel("w1")
plt.show()

However, ratio is maintained between components:

In [ ]:
plt.plot(np.log10(cfs['tol']),cfs['w2']/cfs['w1'])
plt.scatter(np.log10(cfs['tol']),cfs['w2']/cfs['w1'])
plt.xlabel("Tolerance")
plt.ylabel("w2")
plt.show()

## Review Questions

See: @sec-logistic-questions.